## SQLAlchemy

In [3]:
import os                   
import pathlib              

here = pathlib.Path.cwd()   

ROOT = here.parents[2] if here.name == "day04" else here # hanwha-agent

os.chdir(ROOT)              # 루트 디렉토리 설정 : 앞으로 모든 상대경로는 이 폴더가 기준이 된다.

In [5]:
SANDBOX = ROOT / "sandbox" / "w2" / "day04"

# SQLite 
# - 파이썬 표준 라이브러리
# - 별도 설치 필요 없음
import sqlite3

# 실습용 DB 파일 생성 경로 설정
db_path = SANDBOX / "sql_practice.db"

# db 파일 연결  
# - 파일이 없으면 새로 만들어짐
conn = sqlite3.connect(db_path)

# 커서(cursor)
# - DB에 명령을 보내고 결과를 받아오는 창구
cur = conn.cursor()

print(f"연결 완료 : {db_path}")

연결 완료 : c:\workspace\hanwha-agent\sandbox\w2\day04\sql_practice.db


* SQLite 연습

In [12]:
# 테이블 생성
sql = """
CREATE TABLE documents(
    id INTEGER PRIMARY KEY,
    doc_id VARCHAR(20) NOT NULL,
    title VARCHAR(200) NOT NULL,
    version VARCHAR(10) NOT NULL,
    department VARCHAR(15) NOT NULL,
    security_level VARCHAR(10) NOT NULL,
    valid_date DATE NOT NULL,
    expiry_date DATE,
    is_latest BOOLEAN NOT NULL DEFAULT FALSE,
    page_count INTEGER NOT NULL DEFAULT 0,
    create_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
);
"""
# 쿼리문 실행
cur.execute(sql) 

print("DB 테이블 생성")

# 테이블 리스트 조회
cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(f"테이블 목록 : {cur.fetchall()}")

OperationalError: table documents already exists

In [14]:
cur.execute("ALTER TABLE documents RENAME COLUMN expiray_date TO expiry_date;")
conn.commit()
print("컬럼명 수정 완료")

# 확인
cur.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='documents';")
print(cur.fetchone()[0])

컬럼명 수정 완료
CREATE TABLE documents(
    id INTEGER PRIMARY KEY,
    doc_id VARCHAR(20) NOT NULL,
    title VARCHAR(200) NOT NULL,
    version VARCHAR(10) NOT NULL,
    department VARCHAR(15) NOT NULL,
    security_level VARCHAR(10) NOT NULL,
    valid_date DATE NOT NULL,
    expiry_date DATE,
    is_latest BOOLEAN NOT NULL DEFAULT FALSE,
    page_count INTEGER NOT NULL DEFAULT 0,
    create_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
)


In [ ]:
# 데이터 저장
sql = """
INSERT INTO documents(doc_id, title, version, department, security_level, valid_date, expiry_date, is_latest, page_count) 
VALUES(?, ?, ?, ?, ?, ?, ?, ?, ?);
"""
cur.execute(sql, ("DOC-HR-012", "출장 여비 규정", "1.0", "인사팀", "일반", "2025-01-01", "2025-12-31", False, 28))
print("저장 완료")
conn.commit()

저장 완료


In [21]:
# 데이터 조회
sql = "SELECT id, doc_id, title, version, department FROM documents;"
cur.execute(sql, ())

result = cur.fetchall()
print(result)

for r in result:
    # print([str(val) for i, val in enumerate(r)])
    id = r[0]
    doc_id = r[1]
    title = r[2]
    print(id, doc_id, title)

conn.close()

[(1, 'DOC-HR-012', '출장 여비 규정', '1.0', '인사팀')]
1 DOC-HR-012 출장 여비 규정


In [23]:
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker

# engine = create_engine("sqlite:///./sql_practice.db")
engine = create_engine(f"sqlite:///{db_path}")
SessionLocal = sessionmaker(bind=engine)

# with 를 사용하면 close()를 따로 명시하지 않아도 된다.
with SessionLocal() as session:
    rows = session.execute(
        text("SELECT id, doc_id, title, version, department FROM documents")
    ).all()

    session.commit()

In [24]:
conn.close()

In [25]:
engine.dispose()